# BÀI THỰC HÀNH LAB 04: MÔ HÌNH TUYẾN TÍNH & HỒI QUY TUYẾN TÍNH
## Môn học: Xác suất Thống kê (UET.MAT1052)
### Môi trường: Python 3, Pandas, Statsmodels & plotnine

---
**Họ và tên sinh viên:** .....................................................  
**Mã số sinh viên (MSV):** .....................................................  
**Lớp học phần:** .....................................................  

> **Hướng dẫn:** Chạy các ô code bằng tổ hợp phím `Shift + Enter`. Hoàn thành các câu hỏi phân tích nhận xét và các ô code theo yêu cầu.

### 1. Khởi động môi trường và nạp thư viện
Nạp các thư viện tính toán và trực quan hóa dữ liệu chuẩn của Python theo ngữ pháp đồ họa Grammar of Graphics (`plotnine`).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from plotnine import (
    ggplot,
    aes,
    geom_point,
    geom_smooth,
    geom_hline,
    geom_line,
    labs,
    theme_minimal,
    scale_color_manual,
)

### 2. Khám phá dữ liệu và tính hệ số tương quan ($r$) trên 51 Bang Hoa Kỳ
Tập dữ liệu `poverty_grad.csv` ghi nhận tỷ lệ tốt nghiệp THPT (`Graduates`, %) và tỷ lệ nghèo đói (`Poverty`, %) tại 50 bang và Washington D.C.

In [ ]:
# Đọc dữ liệu từ thư mục du-lieu/
df_poverty = pd.read_csv("du-lieu/poverty_grad.csv")
df_poverty.rename(columns={"graduates": "Graduates", "poverty": "Poverty"}, inplace=True)
print(f"Số lượng quan sát: {len(df_poverty)} bang")
df_poverty.head()

In [ ]:
# Tính thống kê mô tả và hệ số tương quan Pearson r
mx = df_poverty["Graduates"].mean()
sx = df_poverty["Graduates"].std()
my = df_poverty["Poverty"].mean()
sy = df_poverty["Poverty"].std()
r = df_poverty["Graduates"].corr(df_poverty["Poverty"])

print(f"Graduates: mean = {mx:.2f}%, std = {sx:.2f}%")
print(f"Poverty:   mean = {my:.2f}%, std = {sy:.2f}%")
print(f"Hệ số tương quan Pearson r = {r:.4f}")

In [ ]:
# Vẽ biểu đồ phân tán kèm đường hồi quy tuyến tính bằng plotnine
p1 = (
    ggplot(df_poverty, aes(x="Graduates", y="Poverty"))
    + geom_point(color="#0284c7", size=3, alpha=0.75)
    + geom_smooth(method="lm", se=True, color="#ea580c", fill="#ea580c", size=1.2)
    + labs(
        title="Biểu đồ phân tán & Đường xu hướng OLS (51 Bang Hoa Kỳ)",
        x="Tỷ lệ tốt nghiệp THPT (%)",
        y="Tỷ lệ sống dưới ngưỡng nghèo (%)",
    )
    + theme_minimal()
)
print(p1)

### 3. Khớp mô hình hồi quy đơn biến OLS bằng `statsmodels`
Áp dụng công thức mô hình hóa: `Poverty ~ Graduates`.

In [ ]:
# Khớp mô hình OLS
model_poverty = smf.ols("Poverty ~ Graduates", data=df_poverty).fit()
print(model_poverty.summary())

### 4. Dự báo và Phân tích phần dư ($e = y - \hat{y}$)
Dự báo cho quan sát mới và kiểm tra phần dư của hai trường hợp cá biệt: Rhode Island và Washington D.C.

In [ ]:
# Dự báo cho bang có tỷ lệ tốt nghiệp 85%
pred_85 = model_poverty.predict(pd.DataFrame({"Graduates": [85.0]}))[0]
print(f"Tỷ lệ nghèo dự báo tại Graduates = 85%: {pred_85:.2f}%")

# Thêm cột fitted và residual vào DataFrame
df_poverty["fitted"] = model_poverty.fittedvalues
df_poverty["residual"] = model_poverty.resid

ri = df_poverty[df_poverty["state"] == "Rhode Island"].iloc[0]
dc = df_poverty[df_poverty["state"] == "District of Columbia"].iloc[0]
print(f"RI: Thực tế = {ri["Poverty"]}%, Khớp = {ri["fitted"]:.2f}%, Phần dư = {ri["residual"]:.2f}%")
print(f"DC: Thực tế = {dc["Poverty"]}%, Khớp = {dc["fitted"]:.2f}%, Phần dư = {dc["residual"]:.2f}%")

In [ ]:
# Vẽ đồ thị phần dư (Residual Plot) bằng plotnine
p_resid = (
    ggplot(df_poverty, aes(x="fitted", y="residual"))
    + geom_point(color="#0284c7", size=3, alpha=0.8)
    + geom_hline(yintercept=0, color="#ea580c", linetype="dashed", size=1)
    + labs(
        title="Đồ thị phần dư (Residuals vs Fitted Values)",
        x="Giá trị khớp ŷ (%)",
        y="Phần dư e (%)",
    )
    + theme_minimal()
)
print(p_resid)

### 5. Hồi quy đa biến với biến phân loại: Mô hình song song trên `books.csv`
Dự báo khối lượng sách (`weight`, g) dựa trên thể tích (`volume`, cm³) và loại bìa (`cover`).

In [ ]:
# Đọc dữ liệu sách
df_books = pd.read_csv("du-lieu/books.csv")
print(f"Số lượng sách: {len(df_books)} cuốn")
print(df_books.head())

# Khớp mô hình song song
model_books = smf.ols("weight ~ volume + C(cover)", data=df_books).fit()
print(model_books.summary())

In [ ]:
# Trích xuất hệ số và viết phương trình hồi quy
b0_b = model_books.params["Intercept"]
b1_b = model_books.params["volume"]
b2_b = model_books.params["C(cover)[T.paperback]"]

print(f"1. Phương trình Bìa cứng (Hardcover): weight_hat = {b0_b:.2f} + {b1_b:.2f} * volume")
print(f"2. Phương trình Bìa mềm (Paperback):  weight_hat = {b0_b + b2_b:.2f} + {b1_b:.2f} * volume")

In [ ]:
# Trực quan hóa hai đường song song bằng plotnine
df_books["fitted_weight"] = model_books.fittedvalues

p_books = (
    ggplot(df_books, aes(x="volume", y="weight", color="cover", shape="cover"))
    + geom_point(size=3.5, alpha=0.85)
    + geom_line(aes(y="fitted_weight", group="cover"), size=1.2)
    + scale_color_manual(values={"hardcover": "#0284c7", "paperback": "#ea580c"})
    + labs(
        title="Mô hình hai đường song song (Parallel Slopes Model)",
        x="Thể tích (cm³)",
        y="Khối lượng (g)",
        color="Loại bìa",
        shape="Loại bìa",
    )
    + theme_minimal()
)
print(p_books)

### 6. Câu hỏi phân tích & Nhận xét (Sinh viên tự trả lời)
1. Trong mô hình nghèo đói, hệ số góc $b_1 = -0.62$ mang ý nghĩa kinh tế - xã hội như thế nào?
2. Tại sao ta không nên dùng mô hình trên để dự báo tỷ lệ nghèo đói cho một địa phương có tỷ lệ tốt nghiệp là 10%?
3. Trong mô hình sách, con số $-184.05$ mang ý nghĩa gì theo nguyên lý Ceteris Paribus?